In [ ]:
# Set up Step 6.
#
# Load the config, mount Drive, prepare input directories, and extract
# Step 5 zip inputs if configured.

import os
import sys
import shutil
import zipfile
import importlib
from pathlib import Path
from google.colab import files
from google.colab import drive


PILOT_ROOT = "/content/pilot_full"
DATA_DIR = os.path.join(PILOT_ROOT, "data")
SCRIPTS_DIR = os.path.join(PILOT_ROOT, "scripts")

os.makedirs(PILOT_ROOT, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(SCRIPTS_DIR, exist_ok=True)

with open(os.path.join(PILOT_ROOT, "__init__.py"), "w", encoding="utf-8") as f:
    f.write("")

if "/content" not in sys.path:
    sys.path.insert(0, "/content")


print("Upload config.py")
uploaded_config = files.upload()

config_upload_name = None

for name in uploaded_config.keys():
    if name.startswith("config") and name.endswith(".py"):
        config_upload_name = name
        break

if config_upload_name is None:
    raise FileNotFoundError(
        f"No config*.py file was uploaded. Uploaded files: {list(uploaded_config.keys())}"
    )

config_target_path = os.path.join(PILOT_ROOT, "config.py")

shutil.copy(
    os.path.join("/content", config_upload_name),
    config_target_path,
)


import pilot_full.config as cfg
importlib.reload(cfg)


drive.mount("/content/drive")


STEP6_OUTPUT_DIR = cfg.STEP6_OUTPUT_DIR
STEP6_INPUT_DIRS = cfg.STEP6_INPUT_DIRS
STEP6_INPUT_ZIP_PATHS = getattr(cfg, "STEP6_INPUT_ZIP_PATHS", {})
STEP6_USE_STEP5_ZIP_INPUT = bool(getattr(cfg, "STEP6_USE_STEP5_ZIP_INPUT", False))

if not isinstance(STEP6_INPUT_DIRS, dict):
    raise TypeError(
        "cfg.STEP6_INPUT_DIRS must be a dict: sample_family -> input_dir. "
        f"Got: {type(STEP6_INPUT_DIRS)}"
    )

if not STEP6_INPUT_DIRS:
    raise ValueError("cfg.STEP6_INPUT_DIRS is empty.")


if os.path.exists(STEP6_OUTPUT_DIR):
    shutil.rmtree(STEP6_OUTPUT_DIR)

os.makedirs(STEP6_OUTPUT_DIR, exist_ok=True)

all_discovered_pt_files = {}

for sample_family, input_dir in STEP6_INPUT_DIRS.items():
    os.makedirs(input_dir, exist_ok=True)

    if STEP6_USE_STEP5_ZIP_INPUT:
        if sample_family not in STEP6_INPUT_ZIP_PATHS:
            raise KeyError(
                f"Missing zip path for sample_family={sample_family!r} "
                "in cfg.STEP6_INPUT_ZIP_PATHS."
            )

        zip_path = STEP6_INPUT_ZIP_PATHS[sample_family]

        if not os.path.exists(zip_path):
            raise FileNotFoundError(
                f"Step5 zip not found for sample_family={sample_family!r}.\n"
                f"Expected path: {zip_path}"
            )

        # Clear stale extracted files for this family.
        for old_path in Path(input_dir).glob("*"):
            if old_path.is_file():
                old_path.unlink()
            elif old_path.is_dir():
                shutil.rmtree(old_path)

        print("\nExtracting Step5 zip")
        print("sample_family:", sample_family)
        print("zip_path:", zip_path)
        print("target_dir:", input_dir)
        print("zip_size_GB:", round(os.path.getsize(zip_path) / 1024**3, 3))

        with zipfile.ZipFile(zip_path, "r") as zf:
            members = zf.infolist()
            total_bytes = sum(m.file_size for m in members)
            extracted_bytes = 0

            print(f"Files in zip: {len(members)}")
            print(f"Uncompressed size GB: {round(total_bytes / 1024**3, 3)}")

            for i, member in enumerate(members, start=1):
                zf.extract(member, input_dir)
                extracted_bytes += member.file_size

                percent = extracted_bytes / total_bytes * 100 if total_bytes > 0 else 100

                print(
                    f"\rExtracting {i}/{len(members)} files "
                    f"({percent:.2f}%) "
                    f"{extracted_bytes / 1024**3:.3f}/{total_bytes / 1024**3:.3f} GB",
                    end="",
                    flush=True,
                )

            print("\nExtraction complete.")

    pt_files = sorted(Path(input_dir).rglob("*.pt"))

    if not pt_files:
        raise FileNotFoundError(
            f"No Step5 .pt files found for sample_family={sample_family!r}.\n"
            f"Directory: {input_dir}\n"
            f"STEP6_USE_STEP5_ZIP_INPUT={STEP6_USE_STEP5_ZIP_INPUT}"
        )

    all_discovered_pt_files[sample_family] = [
        str(p.relative_to(input_dir))
        for p in pt_files
    ]

print("\nSetup complete.")
print("Uploaded config:", config_upload_name)
print("Config copied to:", config_target_path)
print("STEP6_USE_STEP5_ZIP_INPUT:", STEP6_USE_STEP5_ZIP_INPUT)
print("STEP6_OUTPUT_DIR:", STEP6_OUTPUT_DIR)

print("\nStep5 input dirs:")
for sample_family, input_dir in STEP6_INPUT_DIRS.items():
    print(f"- {sample_family}: {input_dir}")
    print(f"  .pt files found: {len(all_discovered_pt_files[sample_family])}")

print("\nDiscovered Step5 .pt files by family:")
for sample_family, names in all_discovered_pt_files.items():
    print(f"\n[{sample_family}]")
    for name in names:
        print("-", name)

Upload config.py


Saving config_positional_context_full_drive.py to config_positional_context_full_drive.py
Mounted at /content/drive

Setup complete.
Uploaded config: config_positional_context_full_drive.py
Config copied to: /content/pilot_full/config.py
STEP6_USE_STEP5_ZIP_INPUT: False
STEP6_OUTPUT_DIR: /content/drive/MyDrive/Colab Notebooks/linear_probe_full/data_prepare_step1_3/data/step6_probe/step6_pilot_full_ithor120_diverse_qwen2_5_3b_pair_explicit_direct_relation_ld_scene_split

Step5 input dirs:
- pair: /content/drive/MyDrive/Colab Notebooks/linear_probe_full/data_prepare_step1_3/data/step5_hs_pair_qwen2_5_3b
  .pt files found: 118

Discovered Step5 .pt files by family:

[pair]
- FloorPlan10_step4_probe_samples_diverse_step5_hs_qwen2_5_3b.pt
- FloorPlan11_step4_probe_samples_diverse_step5_hs_qwen2_5_3b.pt
- FloorPlan12_step4_probe_samples_diverse_step5_hs_qwen2_5_3b.pt
- FloorPlan13_step4_probe_samples_diverse_step5_hs_qwen2_5_3b.pt
- FloorPlan14_step4_probe_samples_diverse_step5_hs_qwen2_5_3b

In [ ]:
# All experiment parameters are loaded from pilot_full.config.

import os
import sys
import json
import random
import importlib
from pathlib import Path
from collections import Counter, defaultdict
from typing import Any, Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import torch

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

if "/content" not in sys.path:
    sys.path.insert(0, "/content")

import pilot_full.config as cfg
importlib.reload(cfg)

# Experiment identity

EXPERIMENT_NAME = cfg.STEP6_EXPERIMENT_NAME
EXPERIMENT_DESCRIPTION = cfg.STEP6_EXPERIMENT_DESCRIPTION
EXPERIMENT_ID = getattr(cfg, "STEP6_EXPERIMENT_ID", EXPERIMENT_NAME)
EXPERIMENT_SHORT_NAME = getattr(cfg, "STEP6_EXPERIMENT_SHORT_NAME", EXPERIMENT_NAME)
SCRIPT_FAMILY = getattr(cfg, "STEP6_SCRIPT_FAMILY", "pair_relation_layerwise_logreg")

SUPPORTED_SCRIPT_FAMILIES = {
    "pair_relation_layerwise_logreg",
}

if SCRIPT_FAMILY not in SUPPORTED_SCRIPT_FAMILIES:
    raise ValueError(
        f"This notebook supports script families {sorted(SUPPORTED_SCRIPT_FAMILIES)}, "
        f"but config requests STEP6_SCRIPT_FAMILY={SCRIPT_FAMILY!r}."
    )

# Core config

RANDOM_SEED = cfg.RANDOM_SEED

MODEL_NAME = cfg.STEP6_MODEL_NAME
MODEL_TAG = cfg.STEP6_MODEL_TAG

SAMPLE_FAMILY = cfg.STEP6_SAMPLE_FAMILY
SAMPLE_FAMILIES = list(getattr(cfg, "STEP6_SAMPLE_FAMILIES", [SAMPLE_FAMILY]))

FEATURE_KEY = cfg.STEP6_FEATURE_KEY
LABEL_FIELD = cfg.STEP6_LABEL_FIELD

EXPLICIT_DIRECT = cfg.EXPLICIT_DIRECT
EXPLICIT_INVERSE = cfg.EXPLICIT_INVERSE
IMPLICIT_LABELED = getattr(cfg, "IMPLICIT_LABELED", "implicit_labeled")
IMPLICIT_NONE = getattr(cfg, "IMPLICIT_NONE", "implicit_none")
IMPLICIT_TRANSITIVE = getattr(cfg, "IMPLICIT_TRANSITIVE", "implicit_transitive")

KEEP_EVIDENCE_TYPES = set(getattr(cfg, "STEP6_KEEP_EVIDENCE_TYPES", set()))
DROP_NONE_LABEL = bool(getattr(cfg, "STEP6_DROP_NONE_LABEL", False))
DROP_EMPTY_LABEL = bool(getattr(cfg, "STEP6_DROP_EMPTY_LABEL", False))
NONE_RELATION_LABEL = getattr(cfg, "NONE_RELATION_LABEL", "none")


STEP6_INPUT_DIRS = cfg.STEP6_INPUT_DIRS
STEP6_INPUT_DIR = getattr(cfg, "STEP6_INPUT_DIR", None)
STEP6_OUTPUT_DIR = cfg.STEP6_OUTPUT_DIR

STEP6_RESULT_JSON = getattr(cfg, "STEP6_RESULT_JSON", None)
STEP6_LAYER_SCORES_CSV = getattr(cfg, "STEP6_LAYER_SCORES_CSV", None)
STEP6_LABEL_METRICS_CSV = getattr(cfg, "STEP6_LABEL_METRICS_CSV", None)
STEP6_RECALL_MATRIX_LONG_CSV = getattr(cfg, "STEP6_RECALL_MATRIX_LONG_CSV", None)
STEP6_TEST_PREDICTIONS_CSV = getattr(cfg, "STEP6_TEST_PREDICTIONS_CSV", None)
STEP6_MANIFEST_JSON = getattr(cfg, "STEP6_MANIFEST_JSON", None)


# Scene split config

TRAIN_SCENES = cfg.STEP6_TRAIN_SCENES
TEST_SCENES = cfg.STEP6_TEST_SCENES
REQUIRE_EXPLICIT_SCENE_SPLIT = cfg.STEP6_REQUIRE_EXPLICIT_SCENE_SPLIT

# Logistic regression config

MAX_ITER = cfg.STEP6_LOGREG_MAX_ITER
C_VALUE = cfg.STEP6_LOGREG_C
CLASS_WEIGHT = getattr(cfg, "STEP6_LOGREG_CLASS_WEIGHT", "balanced")
SOLVER = getattr(cfg, "STEP6_LOGREG_SOLVER", "lbfgs")


PRINT_DATASET_SUMMARY = cfg.STEP6_PRINT_DATASET_SUMMARY
PRINT_LAYER_PROGRESS = cfg.STEP6_PRINT_LAYER_PROGRESS
PRINT_TOP_LAYERS = cfg.STEP6_PRINT_TOP_LAYERS
NUM_TOP_LAYERS_TO_PRINT = cfg.STEP6_NUM_TOP_LAYERS_TO_PRINT


USE_LOCAL_CONTENT_CACHE = bool(
    getattr(cfg, "STEP6_USE_LOCAL_CONTENT_CACHE", False)
)

LOCAL_CONTENT_CACHE_ROOT = getattr(
    cfg,
    "STEP6_LOCAL_CONTENT_CACHE_ROOT",
    f"/content/step5_hs_cache_{MODEL_TAG}_{EXPERIMENT_SHORT_NAME}",
)

LOCAL_CACHE_SAFETY_MARGIN_GB = float(
    getattr(cfg, "STEP6_LOCAL_CACHE_SAFETY_MARGIN_GB", 10.0)
)

LAYER_INDICES_TO_RUN = getattr(
    cfg,
    "STEP6_LAYER_INDICES_TO_RUN",
    None,
)


random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)


if not SAMPLE_FAMILIES:
    raise ValueError("STEP6_SAMPLE_FAMILIES is empty.")

missing_input_families = sorted(set(SAMPLE_FAMILIES) - set(STEP6_INPUT_DIRS.keys()))

if missing_input_families:
    raise KeyError(
        "Some STEP6_SAMPLE_FAMILIES are missing from STEP6_INPUT_DIRS: "
        f"{missing_input_families}"
    )

if not KEEP_EVIDENCE_TYPES:
    raise ValueError(
        "STEP6_KEEP_EVIDENCE_TYPES is empty. "
        "Define evidence types in the active Step6 experiment config."
    )

# This script family is pair-level. It can read one or more family dirs,
# but it expects records to have pair-level fields and relation labels.
if SCRIPT_FAMILY == "pair_relation_layerwise_logreg":
    if "pair" not in SAMPLE_FAMILIES:
        raise ValueError(
            "pair_relation_layerwise_logreg expects 'pair' in STEP6_SAMPLE_FAMILIES. "
            f"Got: {SAMPLE_FAMILIES}"
        )

print("Step 6 config loaded from config.py.")
print("EXPERIMENT_ID:", EXPERIMENT_ID)
print("EXPERIMENT_NAME:", EXPERIMENT_NAME)
print("EXPERIMENT_SHORT_NAME:", EXPERIMENT_SHORT_NAME)
print("EXPERIMENT_DESCRIPTION:", EXPERIMENT_DESCRIPTION)
print("SCRIPT_FAMILY:", SCRIPT_FAMILY)
print("MODEL_NAME:", MODEL_NAME)
print("MODEL_TAG:", MODEL_TAG)
print("SAMPLE_FAMILY:", SAMPLE_FAMILY)
print("SAMPLE_FAMILIES:", SAMPLE_FAMILIES)
print("FEATURE_KEY:", FEATURE_KEY)
print("LABEL_FIELD:", LABEL_FIELD)
print("KEEP_EVIDENCE_TYPES:", sorted(KEEP_EVIDENCE_TYPES))
print("DROP_NONE_LABEL:", DROP_NONE_LABEL)
print("DROP_EMPTY_LABEL:", DROP_EMPTY_LABEL)
print("NONE_RELATION_LABEL:", NONE_RELATION_LABEL)
print("RANDOM_SEED:", RANDOM_SEED)
print("TRAIN_SCENES:", TRAIN_SCENES)
print("TEST_SCENES:", TEST_SCENES)
print("REQUIRE_EXPLICIT_SCENE_SPLIT:", REQUIRE_EXPLICIT_SCENE_SPLIT)
print("MAX_ITER:", MAX_ITER)
print("C_VALUE:", C_VALUE)
print("CLASS_WEIGHT:", CLASS_WEIGHT)
print("SOLVER:", SOLVER)
print("STEP6_INPUT_DIRS:")
print(json.dumps(STEP6_INPUT_DIRS, indent=2, ensure_ascii=False))
print("STEP6_OUTPUT_DIR:", STEP6_OUTPUT_DIR)
print("USE_LOCAL_CONTENT_CACHE:", USE_LOCAL_CONTENT_CACHE)
print("LOCAL_CONTENT_CACHE_ROOT:", LOCAL_CONTENT_CACHE_ROOT)
print("LOCAL_CACHE_SAFETY_MARGIN_GB:", LOCAL_CACHE_SAFETY_MARGIN_GB)
print("LAYER_INDICES_TO_RUN:", LAYER_INDICES_TO_RUN)

Step 6 config loaded from config.py.
EXPERIMENT_ID: pair_explicit_direct_relation_ld
EXPERIMENT_NAME: step6_pilot_full_ithor120_diverse_qwen2_5_3b_pair_explicit_direct_relation_ld_scene_split
EXPERIMENT_SHORT_NAME: pair_ed_rel_ld
EXPERIMENT_DESCRIPTION: scene_split_layerwise_logreg_for_directly_stated_pair_relation_labels
SCRIPT_FAMILY: pair_relation_layerwise_logreg
MODEL_NAME: Qwen/Qwen2.5-3B
MODEL_TAG: qwen2_5_3b
SAMPLE_FAMILY: pair
SAMPLE_FAMILIES: ['pair']
FEATURE_KEY: layer_diff_features
LABEL_FIELD: relation
KEEP_EVIDENCE_TYPES: ['explicit_direct']
DROP_NONE_LABEL: True
DROP_EMPTY_LABEL: True
NONE_RELATION_LABEL: none
RANDOM_SEED: 42
TRAIN_SCENES: ['FloorPlan1', 'FloorPlan2', 'FloorPlan3', 'FloorPlan4', 'FloorPlan5', 'FloorPlan6', 'FloorPlan7', 'FloorPlan8', 'FloorPlan9', 'FloorPlan10', 'FloorPlan11', 'FloorPlan12', 'FloorPlan13', 'FloorPlan14', 'FloorPlan15', 'FloorPlan16', 'FloorPlan17', 'FloorPlan18', 'FloorPlan19', 'FloorPlan20', 'FloorPlan21', 'FloorPlan22', 'FloorPlan23', 

In [ ]:
# Utility functions

def make_json_safe(obj):
    """
    Convert numpy / torch objects into JSON-serializable structures.
    """
    if isinstance(obj, dict):
        return {str(k): make_json_safe(v) for k, v in obj.items()}

    if isinstance(obj, list):
        return [make_json_safe(v) for v in obj]

    if isinstance(obj, tuple):
        return [make_json_safe(v) for v in obj]

    if isinstance(obj, set):
        return sorted(make_json_safe(v) for v in obj)

    if isinstance(obj, np.ndarray):
        return obj.tolist()

    if isinstance(obj, torch.Tensor):
        return obj.detach().cpu().tolist()

    if isinstance(obj, (np.integer,)):
        return int(obj)

    if isinstance(obj, (np.floating,)):
        return float(obj)

    if isinstance(obj, (np.bool_,)):
        return bool(obj)

    return obj


def discover_step5_pt_files() -> List[Dict[str, str]]:
    """
    Discover Step5 .pt files from all configured Step6 input families.

    Returns:
        List of records:
            {
                "sample_family": ...,
                "path": absolute pt path,
                "relative_path": path relative to that family's input_dir,
                "input_dir": family input dir,
            }
    """
    discovered = []

    for sample_family in SAMPLE_FAMILIES:
        input_dir = STEP6_INPUT_DIRS[sample_family]
        pt_paths = sorted(Path(input_dir).rglob("*.pt"))

        if not pt_paths:
            raise FileNotFoundError(
                f"No Step5 .pt files found for sample_family={sample_family!r}.\n"
                f"Input dir: {input_dir}"
            )

        for path in pt_paths:
            discovered.append({
                "sample_family": sample_family,
                "path": str(path),
                "relative_path": os.path.relpath(path, input_dir),
                "input_dir": input_dir,
            })

    if not discovered:
        raise FileNotFoundError(
            "No Step5 .pt files found in any configured STEP6_INPUT_DIRS."
        )

    return discovered


def summarize_counter(values):
    return dict(Counter(values))


def safe_get(meta: Dict[str, Any], key: str, default=None):
    value = meta.get(key, default)
    return value

In [ ]:
# Load Step 5 metadata without storing hidden-state arrays.
#
# Discover the Step 5 .pt files from the configured Drive paths, optionally copy
# them to /content, and scan only the metadata needed for Step 6.

import gc
import time

def keep_record_for_step6(
    rec: Dict[str, Any],
    label_field: str,
    keep_evidence_types: set,
    drop_none_label: bool,
    drop_empty_label: bool,
    none_label: str,
) -> bool:
    evidence_type = rec.get("evidence_type")

    if keep_evidence_types and evidence_type not in keep_evidence_types:
        return False

    label = rec.get(label_field)

    if drop_empty_label:
        if label is None:
            return False
        if isinstance(label, str) and label.strip() == "":
            return False

    if drop_none_label and label == none_label:
        return False

    return True

def remount_google_drive_if_needed():

    from google.colab import drive

    print("Attempting to remount Google Drive...", flush=True)

    try:
        drive.flush_and_unmount()
    except Exception as e:
        print(f"flush_and_unmount warning: {repr(e)}", flush=True)

    time.sleep(5)

    drive.mount("/content/drive", force_remount=True)

    print("Google Drive remounted.", flush=True)


def safe_getsize_with_remount(
    path: str,
    max_retries: int = 3,
    sleep_seconds: int = 5,
) -> int:
    """
    Get file size with retry. Useful when Google Drive mount is unstable.
    """
    last_error = None

    for attempt in range(1, max_retries + 1):
        try:
            return os.path.getsize(path)

        except OSError as e:
            last_error = e

            print(
                f"getsize attempt {attempt}/{max_retries} failed:\n"
                f"  path={path}\n"
                f"  error={repr(e)}",
                flush=True,
            )

            if getattr(e, "errno", None) == 107:
                remount_google_drive_if_needed()

            time.sleep(sleep_seconds)

    raise RuntimeError(
        f"Failed to get file size after {max_retries} attempts.\n"
        f"path={path}\n"
        f"last_error={repr(last_error)}"
    )


def copy_file_with_retry(
    src_path: str,
    dst_path: str,
    expected_size: int,
    max_retries: int = 5,
    sleep_seconds: int = 10,
):

    if os.path.exists(dst_path):
        dst_size = os.path.getsize(dst_path)

        if dst_size == expected_size:
            return "reused"

        print(
            f"Existing cached file has wrong size. Removing partial file:\n"
            f"  dst={dst_path}\n"
            f"  dst_size={dst_size}\n"
            f"  expected_size={expected_size}",
            flush=True,
        )

        os.remove(dst_path)

    last_error = None

    for attempt in range(1, max_retries + 1):
        tmp_path = dst_path + ".tmp"

        try:
            if os.path.exists(tmp_path):
                os.remove(tmp_path)

            shutil.copyfile(src_path, tmp_path)

            copied_size = os.path.getsize(tmp_path)

            if copied_size != expected_size:
                raise IOError(
                    f"Copied file size mismatch: copied={copied_size}, expected={expected_size}"
                )

            os.replace(tmp_path, dst_path)

            return "copied"

        except OSError as e:
            last_error = e

            print(
                f"Copy attempt {attempt}/{max_retries} failed with OSError:\n"
                f"  src={src_path}\n"
                f"  dst={dst_path}\n"
                f"  error={repr(e)}",
                flush=True,
            )

            if os.path.exists(tmp_path):
                try:
                    os.remove(tmp_path)
                except Exception as cleanup_error:
                    print(
                        f"Could not remove temp file {tmp_path}: {repr(cleanup_error)}",
                        flush=True,
                    )

            if getattr(e, "errno", None) == 107:
                remount_google_drive_if_needed()

            time.sleep(sleep_seconds)

        except Exception as e:
            last_error = e

            print(
                f"Copy attempt {attempt}/{max_retries} failed:\n"
                f"  src={src_path}\n"
                f"  dst={dst_path}\n"
                f"  error={repr(e)}",
                flush=True,
            )

            if os.path.exists(tmp_path):
                try:
                    os.remove(tmp_path)
                except Exception:
                    pass

            time.sleep(sleep_seconds)

    raise RuntimeError(
        f"Failed to copy file after {max_retries} attempts.\n"
        f"src={src_path}\n"
        f"dst={dst_path}\n"
        f"expected_size={expected_size}\n"
        f"last_error={repr(last_error)}"
    )

def maybe_copy_step5_files_to_local_cache(
    drive_pt_file_records: List[Dict[str, str]],
) -> List[Dict[str, str]]:

    if not USE_LOCAL_CONTENT_CACHE:
        print("USE_LOCAL_CONTENT_CACHE=False. Using Google Drive .pt paths directly.")
        return drive_pt_file_records

    total_size_bytes = 0

    for item in drive_pt_file_records:
        path = item["path"]

        if os.path.exists(path):
            total_size_bytes += safe_getsize_with_remount(path)

    total_size_gb = total_size_bytes / 1024**3

    disk = shutil.disk_usage("/content")
    free_gb = disk.free / 1024**3

    print("\nLocal cache pre-check")
    print("=" * 100)
    print(f"Total Step5 .pt size: {total_size_gb:.2f} GB")
    print(f"Free local /content disk: {free_gb:.2f} GB")
    print(f"Safety margin: {LOCAL_CACHE_SAFETY_MARGIN_GB:.2f} GB")
    print("Local cache root:", LOCAL_CONTENT_CACHE_ROOT)

    if total_size_gb + LOCAL_CACHE_SAFETY_MARGIN_GB > free_gb:
        raise RuntimeError(
            "Not enough local /content disk space for caching.\n"
            f"Total Step5 .pt size: {total_size_gb:.2f} GB\n"
            f"Free /content disk: {free_gb:.2f} GB\n"
            f"Safety margin: {LOCAL_CACHE_SAFETY_MARGIN_GB:.2f} GB\n"
            "Set STEP6_USE_LOCAL_CONTENT_CACHE = False in config, "
            "or cache only a subset for smoke testing."
        )

    os.makedirs(LOCAL_CONTENT_CACHE_ROOT, exist_ok=True)

    cached_pt_file_records = []
    copied_count = 0
    reused_count = 0
    copied_bytes = 0

    start_time = time.time()

    for file_i, item in enumerate(drive_pt_file_records, start=1):
        src_path = item["path"]
        sample_family = item["sample_family"]
        relative_path = item["relative_path"]

        local_family_dir = os.path.join(
            LOCAL_CONTENT_CACHE_ROOT,
            sample_family,
        )
        os.makedirs(local_family_dir, exist_ok=True)

        dst_path = os.path.join(
            local_family_dir,
            os.path.basename(relative_path),
        )

        src_size = safe_getsize_with_remount(src_path)

        print(
            f"[{file_i}/{len(drive_pt_file_records)}] Preparing cache "
            f"{relative_path} ({src_size / 1024**2:.2f} MB)",
            flush=True,
        )

        status = copy_file_with_retry(
            src_path=src_path,
            dst_path=dst_path,
            expected_size=src_size,
            max_retries=5,
            sleep_seconds=10,
        )

        if status == "copied":
            copied_count += 1
            copied_bytes += src_size

            print(
                f"    copied -> {dst_path}",
                flush=True,
            )

        elif status == "reused":
            reused_count += 1

            print(
                f"    already cached -> {dst_path}",
                flush=True,
            )

        else:
            raise ValueError(f"Unknown copy status: {status}")

        cached_item = dict(item)
        cached_item["drive_path"] = src_path
        cached_item["path"] = dst_path
        cached_item["input_dir"] = local_family_dir
        cached_item["relative_path"] = os.path.basename(relative_path)

        cached_pt_file_records.append(cached_item)

    elapsed = time.time() - start_time

    print("\nLocal Step5 cache ready.")
    print("Cache root:", LOCAL_CONTENT_CACHE_ROOT)
    print("Copied files:", copied_count)
    print("Reused cached files:", reused_count)
    print(f"Copied size: {copied_bytes / 1024**3:.2f} GB")
    print(f"Elapsed: {elapsed / 60:.2f} min")

    return cached_pt_file_records


drive_pt_file_records = discover_step5_pt_files()

print("Discovered canonical Step5 .pt files:", len(drive_pt_file_records))
print("Canonical input dirs:")
for sample_family, input_dir in STEP6_INPUT_DIRS.items():
    print(f"- {sample_family}: {input_dir}")

pt_file_records = maybe_copy_step5_files_to_local_cache(
    drive_pt_file_records=drive_pt_file_records,
)

print("\nActive Step5 .pt files for this run:", len(pt_file_records))
print("First active paths:")
for item in pt_file_records[:5]:
    print("-", item["path"])
if len(pt_file_records) > 5:
    print("...")

all_records_meta = []
source_file_summaries = []

FEATURE_KEYS_TO_DROP = {
    "layer_subject_features",
    "layer_object_features",
    "layer_diff_features",
    "layer_concat_features",
}

NUM_LAYERS = None
FEATURE_DIM = None

raw_record_count_total = 0
kept_record_count_total = 0

print("\nStart scanning Step5 .pt payloads metadata only...")
print("=" * 100)

for file_i, item in enumerate(pt_file_records, start=1):
    pt_path = item["path"]
    drive_path = item.get("drive_path", pt_path)
    sample_family = item["sample_family"]
    input_dir = item["input_dir"]
    relative_path = item["relative_path"]

    file_size_mb = os.path.getsize(pt_path) / 1024**2 if os.path.exists(pt_path) else -1

    print(
        f"\n[{file_i}/{len(pt_file_records)}] Scanning "
        f"sample_family={sample_family} | "
        f"file={relative_path} | "
        f"size={file_size_mb:.2f} MB",
        flush=True,
    )

    try:
        payload = torch.load(pt_path, map_location="cpu")
    except Exception as e:
        raise RuntimeError(
            f"Failed to load Step5 .pt file.\n"
            f"sample_family={sample_family}\n"
            f"relative_path={relative_path}\n"
            f"active_path={pt_path}\n"
            f"drive_path={drive_path}\n"
            f"file_size_mb={file_size_mb:.2f}\n"
            f"Original error: {repr(e)}"
        ) from e

    if "records" not in payload:
        raise KeyError(
            f"Expected key 'records' in Step5 payload, but not found.\n"
            f"sample_family={sample_family}\n"
            f"relative_path={relative_path}\n"
            f"active_path={pt_path}\n"
            f"payload_keys={sorted(list(payload.keys()))}"
        )

    records = payload["records"]
    raw_record_count_total += len(records)

    kept_in_file = 0

    for local_record_idx, rec in enumerate(records):
        if not keep_record_for_step6(
            rec=rec,
            label_field=LABEL_FIELD,
            keep_evidence_types=KEEP_EVIDENCE_TYPES,
            drop_none_label=DROP_NONE_LABEL,
            drop_empty_label=DROP_EMPTY_LABEL,
            none_label=NONE_RELATION_LABEL,
        ):
            continue

        # Infer feature shape from first kept record only.
        if NUM_LAYERS is None or FEATURE_DIM is None:
            if FEATURE_KEY not in rec:
                raise KeyError(
                    f"Feature key {FEATURE_KEY!r} not found in first kept record.\n"
                    f"file={relative_path}\n"
                    f"available_keys={sorted(list(rec.keys()))}"
                )

            feature_arr = np.asarray(rec[FEATURE_KEY])

            if feature_arr.ndim != 2:
                raise ValueError(
                    f"Expected {FEATURE_KEY} to be 2D [num_layers, feature_dim], "
                    f"but got shape={feature_arr.shape} in file={relative_path}"
                )

            NUM_LAYERS = int(feature_arr.shape[0])
            FEATURE_DIM = int(feature_arr.shape[1])

            print(
                f"Detected feature shape: "
                f"NUM_LAYERS={NUM_LAYERS}, FEATURE_DIM={FEATURE_DIM}",
                flush=True,
            )

        # Keep only lightweight metadata.
        meta = {}

        for k, v in rec.items():
            if k in FEATURE_KEYS_TO_DROP:
                continue

            # Avoid storing accidental large tensors or arrays.
            if isinstance(v, torch.Tensor):
                continue
            if isinstance(v, np.ndarray):
                continue

            meta[k] = v

        # Step6-added metadata for streaming reload.
        meta["_step6_sample_family"] = sample_family
        meta["_source_step5_pt_file"] = relative_path
        meta["_source_step5_input_dir"] = input_dir
        meta["_source_step5_absolute_pt_file"] = pt_path
        meta["_source_step5_drive_pt_file"] = drive_path
        meta["_source_local_record_idx"] = int(local_record_idx)

        all_records_meta.append(meta)
        kept_in_file += 1

    kept_record_count_total += kept_in_file

    source_file_summaries.append({
        "sample_family": sample_family,
        "pt_file": relative_path,
        "absolute_pt_file": pt_path,
        "drive_pt_file": drive_path,
        "input_dir": input_dir,
        "source_file": payload.get("source_file"),
        "scene": payload.get("scene"),
        "source_type": payload.get("source_type"),
        "model_name": payload.get("model_name"),
        "model_tag": payload.get("model_tag"),
        "num_records_raw": len(records),
        "num_records_kept_after_step6_filter": kept_in_file,
        "payload_keys": sorted(list(payload.keys())),
        "file_size_mb": file_size_mb,
    })

    print(
        f"    raw_records={len(records)} | "
        f"kept_after_filter={kept_in_file} | "
        f"total_kept={len(all_records_meta)}",
        flush=True,
    )

    del payload
    del records
    gc.collect()

if NUM_LAYERS is None or FEATURE_DIM is None:
    raise ValueError(
        "No valid records found after Step6 filtering. "
        "Could not infer NUM_LAYERS / FEATURE_DIM."
    )

metadata = all_records_meta
y = np.asarray([str(meta[LABEL_FIELD]) for meta in metadata])
label_order = sorted(set(y.tolist()))

print("\n" + "=" * 100)
print("Finished scanning Step5 metadata.")
print("Raw records before filtering:", raw_record_count_total)
print("Filtered records for Step6:", len(metadata))
print("Dropped records:", raw_record_count_total - len(metadata))
print("NUM_LAYERS:", NUM_LAYERS)
print("FEATURE_DIM:", FEATURE_DIM)

print("\nLabel order:")
print(label_order)

print("\nLabel counts:")
print(json.dumps(dict(Counter(y.tolist())), indent=2, ensure_ascii=False))

print("\nEvidence type counts:")
print(json.dumps(dict(Counter(meta.get("evidence_type") for meta in metadata)), indent=2, ensure_ascii=False))

print("\nSample family counts:")
print(json.dumps(dict(Counter(meta.get("_step6_sample_family") for meta in metadata)), indent=2, ensure_ascii=False))

print("\nSource file summaries preview:")
print(json.dumps(make_json_safe(source_file_summaries[:5]), indent=2, ensure_ascii=False))

Discovered canonical Step5 .pt files: 118
Canonical input dirs:
- pair: /content/drive/MyDrive/Colab Notebooks/linear_probe_full/data_prepare_step1_3/data/step5_hs_pair_qwen2_5_3b

Local cache pre-check
Total Step5 .pt size: 108.57 GB
Free local /content disk: 205.38 GB
Safety margin: 20.00 GB
Local cache root: /content/step5_hs_cache_qwen2_5_3b_pair_ed_rel_ld
[1/118] Preparing cache FloorPlan10_step4_probe_samples_diverse_step5_hs_qwen2_5_3b.pt (1742.12 MB)
    copied -> /content/step5_hs_cache_qwen2_5_3b_pair_ed_rel_ld/pair/FloorPlan10_step4_probe_samples_diverse_step5_hs_qwen2_5_3b.pt
[2/118] Preparing cache FloorPlan11_step4_probe_samples_diverse_step5_hs_qwen2_5_3b.pt (1417.30 MB)
Copy attempt 1/5 failed with OSError:
  src=/content/drive/MyDrive/Colab Notebooks/linear_probe_full/data_prepare_step1_3/data/step5_hs_pair_qwen2_5_3b/FloorPlan11_step4_probe_samples_diverse_step5_hs_qwen2_5_3b.pt
  dst=/content/step5_hs_cache_qwen2_5_3b_pair_ed_rel_ld/pair/FloorPlan11_step4_probe_sampl

In [ ]:
# Step 6 filtered metadata summary

if not metadata:
    raise ValueError(
        "No metadata records available after Step6 filtering. "
        "Check STEP6_KEEP_EVIDENCE_TYPES, STEP6_LABEL_FIELD, "
        "STEP6_DROP_NONE_LABEL, and Step5 input files."
    )

after_family_counts = summarize_counter(
    meta.get("_step6_sample_family") for meta in metadata
)

after_evidence_counts = summarize_counter(
    meta.get("evidence_type") for meta in metadata
)

after_label_counts = summarize_counter(
    meta.get(LABEL_FIELD) for meta in metadata
)

remaining_evidence_types = set(after_evidence_counts.keys())
unexpected_evidence_types = remaining_evidence_types - KEEP_EVIDENCE_TYPES

if unexpected_evidence_types:
    raise ValueError(
        "Filtering failed: unexpected evidence types remain.\n"
        f"Expected subset of: {sorted(KEEP_EVIDENCE_TYPES)}\n"
        f"Remaining: {sorted(remaining_evidence_types)}"
    )

print("\n" + "=" * 100)
print("Filtered Step6 metadata summary")
print("=" * 100)
print("Number of filtered records:", len(metadata))
print("Sample family counts:")
print(json.dumps(after_family_counts, indent=2, ensure_ascii=False))
print("Evidence type counts:")
print(json.dumps(after_evidence_counts, indent=2, ensure_ascii=False))
print("Label counts:")
print(json.dumps(after_label_counts, indent=2, ensure_ascii=False))


Filtered Step6 metadata summary
Number of filtered records: 24252
Sample family counts:
{
  "pair": 24252
}
Evidence type counts:
{
  "explicit_direct": 24252
}
Label counts:
{
  "on": 4169,
  "left_of": 4913,
  "below": 2243,
  "supports": 2901,
  "above": 2392,
  "right_of": 4374,
  "near": 2336,
  "contains": 403,
  "in": 521
}


In [ ]:
# Validate filtered Step 5 metadata

required_metadata_keys = {
    "sample_id",
    "pair_group_id",
    "scene",
    "paragraph_id",
    "text",
    LABEL_FIELD,
    "evidence_type",
    "probe_direction_relative_to_text",
    "is_inverse_of_text_relation",
    "_step6_sample_family",
    "_source_step5_pt_file",
    "_source_step5_input_dir",
    "_source_step5_absolute_pt_file",
    "_source_step5_drive_pt_file",
    "_source_local_record_idx",
}

missing_examples = []

for idx, meta in enumerate(metadata):
    missing = [key for key in required_metadata_keys if key not in meta]

    if missing:
        missing_examples.append({
            "record_index": idx,
            "sample_id": meta.get("sample_id"),
            "missing_keys": missing,
        })

if missing_examples:
    raise KeyError(
        "Some metadata records are missing required keys. First examples:\n"
        + json.dumps(missing_examples[:10], indent=2, ensure_ascii=False)
    )

metadata_keys = sorted(set().union(*(m.keys() for m in metadata)))

print("Metadata validation passed.")
print("Number of metadata records:", len(metadata))
print("Feature key to be streamed later:", FEATURE_KEY)
print("Number of layers:", NUM_LAYERS)
print("Feature dim:", FEATURE_DIM)
print("Labels:", label_order)

print("\nMetadata keys preserved:")
print(json.dumps(metadata_keys, indent=2, ensure_ascii=False))

Metadata validation passed.
Number of metadata records: 24252
Feature key to be streamed later: layer_diff_features
Number of layers: 37
Feature dim: 2048
Labels: ['above', 'below', 'contains', 'in', 'left_of', 'near', 'on', 'right_of', 'supports']

Metadata keys preserved:
[
  "_source_local_record_idx",
  "_source_step5_absolute_pt_file",
  "_source_step5_drive_pt_file",
  "_source_step5_input_dir",
  "_source_step5_pt_file",
  "_step6_sample_family",
  "bridge_uids",
  "chunk_id",
  "cluster_id",
  "composition_type",
  "decoded_tokens",
  "evidence_type",
  "generation_mode",
  "geometry_label",
  "has_relation_label",
  "implicit_rules",
  "is_inverse_of_text_relation",
  "label_is_none",
  "max_length",
  "num_supporting_paths",
  "num_tokens",
  "object_alias",
  "object_all_char_spans",
  "object_id",
  "object_token_spans",
  "object_type",
  "object_uid",
  "offset_mapping",
  "pair_evidence_type",
  "pair_group_id",
  "paragraph_id",
  "paragraph_index_within_cluster",
  "pr

In [ ]:
# Streaming feature loader

def build_records_by_file(metadata: List[Dict[str, Any]]) -> Dict[str, List[Tuple[int, int]]]:
    """
    Return:
        absolute_pt_path -> list of (global_metadata_index, local_record_idx)
    """
    records_by_file = defaultdict(list)

    for global_idx, meta in enumerate(metadata):
        pt_path = meta["_source_step5_absolute_pt_file"]
        local_record_idx = int(meta["_source_local_record_idx"])

        records_by_file[pt_path].append(
            (global_idx, local_record_idx)
        )

    return records_by_file


RECORDS_BY_FILE = build_records_by_file(metadata)

print("Prepared streaming index.")
print("Number of source .pt files used:", len(RECORDS_BY_FILE))

first_active_path = next(iter(RECORDS_BY_FILE.keys()))
print("First active .pt path:", first_active_path)

if USE_LOCAL_CONTENT_CACHE:
    assert first_active_path.startswith(LOCAL_CONTENT_CACHE_ROOT), (
        "USE_LOCAL_CONTENT_CACHE=True, but active .pt path is not under "
        f"LOCAL_CONTENT_CACHE_ROOT={LOCAL_CONTENT_CACHE_ROOT!r}.\n"
        f"First active path: {first_active_path}"
    )
    print("Local /content cache path check passed.")


def load_features_for_one_layer(
    layer_idx: int,
    feature_key: str,
    metadata: List[Dict[str, Any]],
    records_by_file: Dict[str, List[Tuple[int, int]]],
) -> np.ndarray:
    """
    Load one layer's features for all filtered metadata records.

    Returns:
        X: np.ndarray with shape [num_examples, feature_dim]

    Memory behavior:
        Preallocates X_layer and fills rows directly.
        This avoids holding an additional dict of row arrays.
    """

    X_layer = np.empty(
        (len(metadata), FEATURE_DIM),
        dtype=np.float32,
    )

    filled = np.zeros(
        len(metadata),
        dtype=bool,
    )

    total_files = len(records_by_file)

    for file_i, (pt_path, index_pairs) in enumerate(records_by_file.items(), start=1):
        relative_name = metadata[index_pairs[0][0]].get(
            "_source_step5_pt_file",
            pt_path,
        )

        print(
            f"    layer={layer_idx} | loading file "
            f"[{file_i}/{total_files}] {relative_name} | "
            f"records_needed={len(index_pairs)}",
            flush=True,
        )

        try:
            payload = torch.load(pt_path, map_location="cpu")
        except Exception as e:
            raise RuntimeError(
                f"Failed to load Step5 .pt while streaming features.\n"
                f"layer_idx={layer_idx}\n"
                f"pt_path={pt_path}\n"
                f"Original error: {repr(e)}"
            ) from e

        if "records" not in payload:
            raise KeyError(
                f"Expected key 'records' in Step5 payload, but not found.\n"
                f"pt_path={pt_path}\n"
                f"payload_keys={sorted(list(payload.keys()))}"
            )

        records = payload["records"]

        for global_idx, local_record_idx in index_pairs:
            rec = records[local_record_idx]

            if feature_key not in rec:
                raise KeyError(
                    f"Feature key {feature_key!r} not found in record.\n"
                    f"pt_path={pt_path}\n"
                    f"local_record_idx={local_record_idx}\n"
                    f"available_keys={sorted(list(rec.keys()))}"
                )

            feat = rec[feature_key]

            if isinstance(feat, torch.Tensor):
                # Extract only the requested layer, not the full [layers, dim] matrix as float32.
                row = feat[layer_idx].detach().cpu().float().numpy()
            else:
                row = np.asarray(feat[layer_idx], dtype=np.float32)

            if row.shape != (FEATURE_DIM,):
                raise ValueError(
                    f"Unexpected layer feature shape: {row.shape}, "
                    f"expected {(FEATURE_DIM,)}.\n"
                    f"pt_path={pt_path}\n"
                    f"local_record_idx={local_record_idx}\n"
                    f"layer_idx={layer_idx}"
                )

            X_layer[global_idx] = row
            filled[global_idx] = True

        del payload
        del records
        gc.collect()

    missing_indices = np.where(~filled)[0].tolist()

    if missing_indices:
        raise ValueError(
            f"Missing features for {len(missing_indices)} metadata rows. "
            f"First missing indices: {missing_indices[:10]}"
        )

    return X_layer


print("Streaming feature loader ready.")

Prepared streaming index.
Number of source .pt files used: 118
First active .pt path: /content/step5_hs_cache_qwen2_5_3b_pair_ed_rel_ld/pair/FloorPlan10_step4_probe_samples_diverse_step5_hs_qwen2_5_3b.pt
Local /content cache path check passed.
Streaming feature loader ready.


In [ ]:
# Scene-based train/test split
#
# Train:
#   filtered Step6 samples from STEP6_TRAIN_SCENES
#
# Test:
#   filtered Step6 samples from STEP6_TEST_SCENES

available_scenes = sorted(set(meta["scene"] for meta in metadata))

if TRAIN_SCENES is None or TEST_SCENES is None:
    if REQUIRE_EXPLICIT_SCENE_SPLIT:
        raise ValueError(
            "STEP6_TRAIN_SCENES and STEP6_TEST_SCENES must be defined in config.py."
        )
    else:
        raise ValueError(
            "Automatic scene split is disabled in this notebook version. "
            "Please define STEP6_TRAIN_SCENES and STEP6_TEST_SCENES in config.py."
        )

ACTIVE_TRAIN_SCENES = list(TRAIN_SCENES)
ACTIVE_TEST_SCENES = list(TEST_SCENES)
SCENE_SPLIT_SOURCE = "config"

train_scene_set = set(ACTIVE_TRAIN_SCENES)
test_scene_set = set(ACTIVE_TEST_SCENES)

overlap = train_scene_set & test_scene_set

if overlap:
    raise ValueError(
        f"Train/test scenes overlap: {sorted(overlap)}"
    )

missing_train_scenes = sorted(train_scene_set - set(available_scenes))
missing_test_scenes = sorted(test_scene_set - set(available_scenes))

print("Available scenes in filtered Step6 records:", len(available_scenes))
print("Available scenes:", available_scenes)

if missing_train_scenes:
    print("Warning: train scenes with no filtered records:", missing_train_scenes)

if missing_test_scenes:
    print("Warning: test scenes with no filtered records:", missing_test_scenes)

train_idx = np.asarray(
    [i for i, meta in enumerate(metadata) if meta["scene"] in train_scene_set],
    dtype=np.int64,
)

test_idx = np.asarray(
    [i for i, meta in enumerate(metadata) if meta["scene"] in test_scene_set],
    dtype=np.int64,
)

if len(train_idx) == 0:
    raise ValueError("No training examples after scene split.")

if len(test_idx) == 0:
    raise ValueError("No test examples after scene split.")

train_label_counts = dict(Counter(y[train_idx].tolist()))
test_label_counts = dict(Counter(y[test_idx].tolist()))

scene_split_info = {
    "scene_split_source": SCENE_SPLIT_SOURCE,
    "train_scenes": ACTIVE_TRAIN_SCENES,
    "test_scenes": ACTIVE_TEST_SCENES,
    "available_scenes": available_scenes,
    "missing_train_scenes": missing_train_scenes,
    "missing_test_scenes": missing_test_scenes,
    "num_train_examples": int(len(train_idx)),
    "num_test_examples": int(len(test_idx)),
    "train_label_counts": train_label_counts,
    "test_label_counts": test_label_counts,
}

print("\nScene split complete.")
print(json.dumps(make_json_safe(scene_split_info), indent=2, ensure_ascii=False))

Available scenes in filtered Step6 records: 118
Available scenes: ['FloorPlan1', 'FloorPlan10', 'FloorPlan11', 'FloorPlan12', 'FloorPlan13', 'FloorPlan14', 'FloorPlan15', 'FloorPlan16', 'FloorPlan17', 'FloorPlan18', 'FloorPlan19', 'FloorPlan2', 'FloorPlan20', 'FloorPlan201', 'FloorPlan202', 'FloorPlan203', 'FloorPlan204', 'FloorPlan205', 'FloorPlan206', 'FloorPlan207', 'FloorPlan208', 'FloorPlan209', 'FloorPlan21', 'FloorPlan210', 'FloorPlan211', 'FloorPlan212', 'FloorPlan213', 'FloorPlan214', 'FloorPlan215', 'FloorPlan216', 'FloorPlan217', 'FloorPlan218', 'FloorPlan219', 'FloorPlan22', 'FloorPlan220', 'FloorPlan221', 'FloorPlan222', 'FloorPlan223', 'FloorPlan224', 'FloorPlan225', 'FloorPlan226', 'FloorPlan227', 'FloorPlan228', 'FloorPlan229', 'FloorPlan23', 'FloorPlan230', 'FloorPlan24', 'FloorPlan25', 'FloorPlan26', 'FloorPlan27', 'FloorPlan28', 'FloorPlan29', 'FloorPlan3', 'FloorPlan30', 'FloorPlan301', 'FloorPlan302', 'FloorPlan303', 'FloorPlan304', 'FloorPlan305', 'FloorPlan306', 

In [ ]:
# Linear probe training and evaluation functions

def train_probe_for_layer(
    X_train: np.ndarray,
    y_train: np.ndarray,
):
    clf = make_pipeline(
        StandardScaler(),
        LogisticRegression(
            max_iter=MAX_ITER,
            C=C_VALUE,
            class_weight=CLASS_WEIGHT,
            solver=SOLVER,
            multi_class="auto",
            random_state=RANDOM_SEED,
        ),
    )

    clf.fit(X_train, y_train)

    return clf


def evaluate_predictions(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    label_order: List[str],
) -> Dict[str, Any]:
    acc = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)

    report = classification_report(
        y_true,
        y_pred,
        labels=label_order,
        output_dict=True,
        zero_division=0,
    )

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=label_order,
    )

    return {
        "accuracy": float(acc),
        "macro_f1": float(macro_f1),
        "classification_report": report,
        "confusion_matrix": cm,
        "num_examples": int(len(y_true)),
    }


def subset_indices_from_test(
    base_test_idx: np.ndarray,
    evidence_type: str,
) -> np.ndarray:
    subset = [
        idx for idx in base_test_idx
        if metadata[idx].get("evidence_type") == evidence_type
    ]

    return np.asarray(subset, dtype=np.int64)


def append_label_metrics_rows(
    rows: List[Dict[str, Any]],
    result: Dict[str, Any],
    layer_idx: int,
    subset_name: str,
):
    report = result["classification_report"]

    for label in label_order:
        label_metrics = report.get(label, {})

        rows.append({
            "experiment_name": EXPERIMENT_NAME,
            "experiment_id": EXPERIMENT_ID,
            "script_family": SCRIPT_FAMILY,
            "subset": subset_name,
            "layer": layer_idx,
            "label": label,
            "precision": float(label_metrics.get("precision", 0.0)),
            "recall": float(label_metrics.get("recall", 0.0)),
            "f1_score": float(label_metrics.get("f1-score", 0.0)),
            "support": int(label_metrics.get("support", 0)),
            "accuracy": float(result["accuracy"]),
            "macro_f1": float(result["macro_f1"]),
            "num_examples": int(result["num_examples"]),
        })


def append_recall_matrix_rows(
    rows: List[Dict[str, Any]],
    result: Dict[str, Any],
    layer_idx: int,
    subset_name: str,
):
    report = result["classification_report"]

    for label in label_order:
        label_metrics = report.get(label, {})

        rows.append({
            "experiment_name": EXPERIMENT_NAME,
            "experiment_id": EXPERIMENT_ID,
            "script_family": SCRIPT_FAMILY,
            "subset": subset_name,
            "layer": layer_idx,
            "label": label,
            "metric": "recall",
            "value": float(label_metrics.get("recall", 0.0)),
        })


def append_prediction_rows(
    rows: List[Dict[str, Any]],
    layer_idx: int,
    indices: np.ndarray,
    y_true: np.ndarray,
    y_pred: np.ndarray,
):
    for local_i, global_idx in enumerate(indices):
        meta = metadata[int(global_idx)]

        row = {
            "experiment_name": EXPERIMENT_NAME,
            "experiment_id": EXPERIMENT_ID,
            "script_family": SCRIPT_FAMILY,
            "layer": int(layer_idx),

            "sample_id": meta.get("sample_id"),
            "pair_group_id": meta.get("pair_group_id"),
            "scene": meta.get("scene"),
            "paragraph_id": meta.get("paragraph_id"),

            "evidence_type": meta.get("evidence_type"),
            "probe_direction_relative_to_text": meta.get("probe_direction_relative_to_text"),
            "is_inverse_of_text_relation": meta.get("is_inverse_of_text_relation"),

            "true_label": str(y_true[local_i]),
            "pred_label": str(y_pred[local_i]),
            "is_correct": bool(y_true[local_i] == y_pred[local_i]),

            "text_relation_label": meta.get("text_relation_label"),
            "text_subject_uid": meta.get("text_subject_uid"),
            "text_object_uid": meta.get("text_object_uid"),

            "subject_uid": meta.get("subject_uid"),
            "object_uid": meta.get("object_uid"),
            "subject_alias": meta.get("subject_alias"),
            "object_alias": meta.get("object_alias"),

            "evidence_sentence": meta.get("evidence_sentence"),
            "text": meta.get("text"),

            "_step6_sample_family": meta.get("_step6_sample_family"),
            "_source_step5_pt_file": meta.get("_source_step5_pt_file"),
            "_source_step5_input_dir": meta.get("_source_step5_input_dir"),
        }

        rows.append(row)

In [ ]:
# Layer-wise Step6 probing with streaming feature loading

test_subsets = {
    "overall": test_idx,
}

for evidence_type in sorted(KEEP_EVIDENCE_TYPES):
    subset_name = str(evidence_type)
    subset_idx = subset_indices_from_test(
        base_test_idx=test_idx,
        evidence_type=evidence_type,
    )

    if len(subset_idx) > 0:
        test_subsets[subset_name] = subset_idx

print("Test subset sizes:")
for subset_name, subset_idx in test_subsets.items():
    print(f"{subset_name}: {len(subset_idx)}")

if len(test_subsets["overall"]) != len(test_idx):
    raise ValueError("Internal error: overall test subset does not match test_idx.")

layer_results = []
per_layer_label_metrics = []
recall_matrix_long_rows = []
prediction_rows = []

if LAYER_INDICES_TO_RUN is None:
    LAYER_INDICES_EVALUATED = list(range(NUM_LAYERS))
else:
    LAYER_INDICES_EVALUATED = [int(layer_idx) for layer_idx in LAYER_INDICES_TO_RUN]

invalid_layers = [
    layer_idx for layer_idx in LAYER_INDICES_EVALUATED
    if layer_idx < 0 or layer_idx >= NUM_LAYERS
]

if invalid_layers:
    raise ValueError(
        f"Invalid layer indices requested: {invalid_layers}. "
        f"Valid range is 0..{NUM_LAYERS - 1}."
    )

print("Layer indices to evaluate:", LAYER_INDICES_EVALUATED)

for layer_idx in LAYER_INDICES_EVALUATED:
    print("\n" + "=" * 100)
    print(f"Layer {layer_idx}/{NUM_LAYERS - 1}")
    print("=" * 100)

    X_layer = load_features_for_one_layer(
        layer_idx=layer_idx,
        feature_key=FEATURE_KEY,
        metadata=metadata,
        records_by_file=RECORDS_BY_FILE,
    )

    print(
        f"Loaded X_layer shape={X_layer.shape}, "
        f"memory_MB={X_layer.nbytes / 1024**2:.2f}",
        flush=True,
    )

    clf = train_probe_for_layer(
        X_train=X_layer[train_idx],
        y_train=y[train_idx],
    )

    train_pred = clf.predict(X_layer[train_idx])

    train_result = evaluate_predictions(
        y_true=y[train_idx],
        y_pred=train_pred,
        label_order=label_order,
    )

    layer_row = {
        "layer": int(layer_idx),
        "train": train_result,
    }

    for subset_name, subset_idx in test_subsets.items():
        subset_pred = clf.predict(X_layer[subset_idx])

        subset_result = evaluate_predictions(
            y_true=y[subset_idx],
            y_pred=subset_pred,
            label_order=label_order,
        )

        layer_row[subset_name] = subset_result

        append_label_metrics_rows(
            rows=per_layer_label_metrics,
            result=subset_result,
            layer_idx=layer_idx,
            subset_name=subset_name,
        )

        append_recall_matrix_rows(
            rows=recall_matrix_long_rows,
            result=subset_result,
            layer_idx=layer_idx,
            subset_name=subset_name,
        )

        if subset_name == "overall":
            append_prediction_rows(
                rows=prediction_rows,
                layer_idx=layer_idx,
                indices=subset_idx,
                y_true=y[subset_idx],
                y_pred=subset_pred,
            )

    layer_results.append(layer_row)

    print(
        f"Layer {layer_idx} done | "
        f"overall_macro_f1={layer_row['overall']['macro_f1']:.4f} | "
        f"overall_acc={layer_row['overall']['accuracy']:.4f}",
        flush=True,
    )

    del X_layer
    del clf
    gc.collect()

print("\nLayer-wise streaming probing complete.")
print("Number of layers evaluated:", len(layer_results))
print("Layer indices evaluated:", LAYER_INDICES_EVALUATED)
print("Evaluation subsets:", sorted(test_subsets.keys()))

Test subset sizes:
overall: 4257
explicit_direct: 4257
Layer indices to evaluate: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36]

Layer 0/36
    layer=0 | loading file [1/118] FloorPlan10_step4_probe_samples_diverse_step5_hs_qwen2_5_3b.pt | records_needed=298
    layer=0 | loading file [2/118] FloorPlan11_step4_probe_samples_diverse_step5_hs_qwen2_5_3b.pt | records_needed=278
    layer=0 | loading file [3/118] FloorPlan12_step4_probe_samples_diverse_step5_hs_qwen2_5_3b.pt | records_needed=231
    layer=0 | loading file [4/118] FloorPlan13_step4_probe_samples_diverse_step5_hs_qwen2_5_3b.pt | records_needed=463
    layer=0 | loading file [5/118] FloorPlan14_step4_probe_samples_diverse_step5_hs_qwen2_5_3b.pt | records_needed=303
    layer=0 | loading file [6/118] FloorPlan15_step4_probe_samples_diverse_step5_hs_qwen2_5_3b.pt | records_needed=235
    layer=0 | loading file [7/118] FloorPlan16_step4_

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Layer 0 done | overall_macro_f1=0.4349 | overall_acc=0.4848

Layer 1/36
    layer=1 | loading file [1/118] FloorPlan10_step4_probe_samples_diverse_step5_hs_qwen2_5_3b.pt | records_needed=298
    layer=1 | loading file [2/118] FloorPlan11_step4_probe_samples_diverse_step5_hs_qwen2_5_3b.pt | records_needed=278
    layer=1 | loading file [3/118] FloorPlan12_step4_probe_samples_diverse_step5_hs_qwen2_5_3b.pt | records_needed=231
    layer=1 | loading file [4/118] FloorPlan13_step4_probe_samples_diverse_step5_hs_qwen2_5_3b.pt | records_needed=463
    layer=1 | loading file [5/118] FloorPlan14_step4_probe_samples_diverse_step5_hs_qwen2_5_3b.pt | records_needed=303
    layer=1 | loading file [6/118] FloorPlan15_step4_probe_samples_diverse_step5_hs_qwen2_5_3b.pt | records_needed=235
    layer=1 | loading file [7/118] FloorPlan16_step4_probe_samples_diverse_step5_hs_qwen2_5_3b.pt | records_needed=337
    layer=1 | loading file [8/118] FloorPlan17_step4_probe_samples_diverse_step5_hs_qwen2_5_3b.

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Layer 1 done | overall_macro_f1=0.8116 | overall_acc=0.8323

Layer 2/36
    layer=2 | loading file [1/118] FloorPlan10_step4_probe_samples_diverse_step5_hs_qwen2_5_3b.pt | records_needed=298
    layer=2 | loading file [2/118] FloorPlan11_step4_probe_samples_diverse_step5_hs_qwen2_5_3b.pt | records_needed=278
    layer=2 | loading file [3/118] FloorPlan12_step4_probe_samples_diverse_step5_hs_qwen2_5_3b.pt | records_needed=231
    layer=2 | loading file [4/118] FloorPlan13_step4_probe_samples_diverse_step5_hs_qwen2_5_3b.pt | records_needed=463
    layer=2 | loading file [5/118] FloorPlan14_step4_probe_samples_diverse_step5_hs_qwen2_5_3b.pt | records_needed=303
    layer=2 | loading file [6/118] FloorPlan15_step4_probe_samples_diverse_step5_hs_qwen2_5_3b.pt | records_needed=235
    layer=2 | loading file [7/118] FloorPlan16_step4_probe_samples_diverse_step5_hs_qwen2_5_3b.pt | records_needed=337
    layer=2 | loading file [8/118] FloorPlan17_step4_probe_samples_diverse_step5_hs_qwen2_5_3b.

...

Layer 34 done | overall_macro_f1=0.7135 | overall_acc=0.7066

Layer 35/36
    layer=35 | loading file [1/118] FloorPlan10_step4_probe_samples_diverse_step5_hs_qwen2_5_3b.pt | records_needed=298
    layer=35 | loading file [2/118] FloorPlan11_step4_probe_samples_diverse_step5_hs_qwen2_5_3b.pt | records_needed=278
    layer=35 | loading file [3/118] FloorPlan12_step4_probe_samples_diverse_step5_hs_qwen2_5_3b.pt | records_needed=231
    layer=35 | loading file [4/118] FloorPlan13_step4_probe_samples_diverse_step5_hs_qwen2_5_3b.pt | records_needed=463
    layer=35 | loading file [5/118] FloorPlan14_step4_probe_samples_diverse_step5_hs_qwen2_5_3b.pt | records_needed=303
    layer=35 | loading file [6/118] FloorPlan15_step4_probe_samples_diverse_step5_hs_qwen2_5_3b.pt | records_needed=235
    layer=35 | loading file [7/118] FloorPlan16_step4_probe_samples_diverse_step5_hs_qwen2_5_3b.pt | records_needed=337
    layer=35 | loading file [8/118] FloorPlan17_step4_probe_samples_diverse_step5_hs_q

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Layer 35 done | overall_macro_f1=0.7198 | overall_acc=0.7021

Layer 36/36
    layer=36 | loading file [1/118] FloorPlan10_step4_probe_samples_diverse_step5_hs_qwen2_5_3b.pt | records_needed=298
    layer=36 | loading file [2/118] FloorPlan11_step4_probe_samples_diverse_step5_hs_qwen2_5_3b.pt | records_needed=278
    layer=36 | loading file [3/118] FloorPlan12_step4_probe_samples_diverse_step5_hs_qwen2_5_3b.pt | records_needed=231
    layer=36 | loading file [4/118] FloorPlan13_step4_probe_samples_diverse_step5_hs_qwen2_5_3b.pt | records_needed=463
    layer=36 | loading file [5/118] FloorPlan14_step4_probe_samples_diverse_step5_hs_qwen2_5_3b.pt | records_needed=303
    layer=36 | loading file [6/118] FloorPlan15_step4_probe_samples_diverse_step5_hs_qwen2_5_3b.pt | records_needed=235
    layer=36 | loading file [7/118] FloorPlan16_step4_probe_samples_diverse_step5_hs_qwen2_5_3b.pt | records_needed=337
    layer=36 | loading file [8/118] FloorPlan17_step4_probe_samples_diverse_step5_hs_q

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Layer 36 done | overall_macro_f1=0.7133 | overall_acc=0.7068

Layer-wise streaming probing complete.
Number of layers evaluated: 37
Layer indices evaluated: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36]
Evaluation subsets: ['explicit_direct', 'overall']


In [ ]:
# Save Step 6 outputs

if not layer_results:
    raise ValueError(
        "layer_results is empty. Please run the layer-wise training/evaluation cell first."
    )

def get_layer_metric(row, subset_key, metric_key):
    value = row.get(subset_key, {}).get(metric_key, None)

    if value is None:
        return float("-inf")

    return float(value)


best_layer_summary = {}

for subset_name in sorted(test_subsets.keys()):
    best_layer = max(
        layer_results,
        key=lambda row: get_layer_metric(row, subset_name, "macro_f1"),
    )

    best_layer_summary[f"best_{subset_name}_macro_f1"] = {
        "layer": int(best_layer["layer"]),
        "macro_f1": float(best_layer[subset_name]["macro_f1"]),
        "accuracy": float(best_layer[subset_name]["accuracy"]),
    }

layer_score_rows = []

for row in layer_results:
    layer_score_row = {
        "experiment_name": EXPERIMENT_NAME,
        "experiment_id": EXPERIMENT_ID,
        "script_family": SCRIPT_FAMILY,
        "layer": int(row["layer"]),

        "train_accuracy": float(row["train"]["accuracy"]),
        "train_macro_f1": float(row["train"]["macro_f1"]),
    }

    for subset_name in sorted(test_subsets.keys()):
        subset_result = row[subset_name]

        layer_score_row[f"{subset_name}_test_accuracy"] = float(subset_result["accuracy"])
        layer_score_row[f"{subset_name}_test_macro_f1"] = float(subset_result["macro_f1"])
        layer_score_row[f"{subset_name}_num_examples"] = int(subset_result["num_examples"])

    layer_score_rows.append(layer_score_row)

layer_scores_df = pd.DataFrame(layer_score_rows)
metrics_df = pd.DataFrame(per_layer_label_metrics)
recall_matrix_long_df = pd.DataFrame(recall_matrix_long_rows)
predictions_df = pd.DataFrame(prediction_rows)

merged_record_summary = {
    "num_records_before_filtering": int(raw_record_count_total),
    "num_records_after_filtering": int(len(metadata)),
    "num_records_dropped_by_step6_filter": int(raw_record_count_total - len(metadata)),
    "sample_family_counts": dict(Counter(meta.get("_step6_sample_family") for meta in metadata)),
    "evidence_type_counts": dict(Counter(meta.get("evidence_type") for meta in metadata)),
    "label_counts": dict(Counter(y.tolist())),
    "metadata_keys": sorted(set().union(*(m.keys() for m in metadata))),
    "feature_loading_mode": "streaming_by_layer",
    "use_local_content_cache": bool(USE_LOCAL_CONTENT_CACHE),
    "local_content_cache_root": LOCAL_CONTENT_CACHE_ROOT if USE_LOCAL_CONTENT_CACHE else None,
}

step6_config_summary = {
    "experiment_id": EXPERIMENT_ID,
    "experiment_name": EXPERIMENT_NAME,
    "experiment_short_name": EXPERIMENT_SHORT_NAME,
    "experiment_description": EXPERIMENT_DESCRIPTION,
    "script_family": SCRIPT_FAMILY,

    "model_name": MODEL_NAME,
    "model_tag": MODEL_TAG,

    "sample_family": SAMPLE_FAMILY,
    "sample_families": SAMPLE_FAMILIES,

    "feature_key": FEATURE_KEY,
    "label_field": LABEL_FIELD,

    "keep_evidence_types": sorted(KEEP_EVIDENCE_TYPES),
    "drop_none_label": DROP_NONE_LABEL,
    "drop_empty_label": DROP_EMPTY_LABEL,
    "none_relation_label": NONE_RELATION_LABEL,

    "logreg_max_iter": MAX_ITER,
    "logreg_C": C_VALUE,
    "logreg_class_weight": CLASS_WEIGHT,
    "logreg_solver": SOLVER,

    "random_seed": RANDOM_SEED,

    "step6_input_dirs": STEP6_INPUT_DIRS,
    "step6_output_dir": STEP6_OUTPUT_DIR,

    "use_local_content_cache": USE_LOCAL_CONTENT_CACHE,
    "local_content_cache_root": LOCAL_CONTENT_CACHE_ROOT if USE_LOCAL_CONTENT_CACHE else None,
    "local_cache_safety_margin_gb": LOCAL_CACHE_SAFETY_MARGIN_GB,

    "layer_indices_requested": LAYER_INDICES_TO_RUN,
    "layer_indices_evaluated": LAYER_INDICES_EVALUATED,
}

full_result = {
    "experiment_id": EXPERIMENT_ID,
    "experiment_name": EXPERIMENT_NAME,
    "description": EXPERIMENT_DESCRIPTION,
    "step6_config": step6_config_summary,

    "source_step5_files": [
        {
           "sample_family": item["sample_family"],
           "relative_path": item["relative_path"],
           "input_dir": item["input_dir"],
           "active_path": item["path"],
           "drive_path": item.get("drive_path", item["path"]),
        }
        for item in pt_file_records
      ],

    "source_file_summaries": source_file_summaries,

    "merged_record_summary": merged_record_summary,
    "scene_split_info": scene_split_info,

    "label_order": label_order,
    "num_layers_total": NUM_LAYERS,
    "num_layers_evaluated": len(LAYER_INDICES_EVALUATED),
    "layer_indices_evaluated": LAYER_INDICES_EVALUATED,
    "feature_dim": FEATURE_DIM,

    "test_subsets": {
        name: int(len(indices))
        for name, indices in test_subsets.items()
    },

    "best_layer_summary": best_layer_summary,
    "results_by_layer": layer_results,
    "per_layer_label_metrics": per_layer_label_metrics,
    "recall_matrix_long": recall_matrix_long_rows,
}

# Prefer config-defined output paths.
result_json_path = STEP6_RESULT_JSON or os.path.join(
    STEP6_OUTPUT_DIR,
    f"{EXPERIMENT_NAME}_full_results.json",
)

layer_scores_csv_path = STEP6_LAYER_SCORES_CSV or os.path.join(
    STEP6_OUTPUT_DIR,
    f"{EXPERIMENT_NAME}_layer_scores.csv",
)

metrics_csv_path = STEP6_LABEL_METRICS_CSV or os.path.join(
    STEP6_OUTPUT_DIR,
    f"{EXPERIMENT_NAME}_per_layer_label_metrics.csv",
)

recall_matrix_long_csv_path = STEP6_RECALL_MATRIX_LONG_CSV or os.path.join(
    STEP6_OUTPUT_DIR,
    f"{EXPERIMENT_NAME}_recall_matrix_long.csv",
)

predictions_csv_path = STEP6_TEST_PREDICTIONS_CSV or os.path.join(
    STEP6_OUTPUT_DIR,
    f"{EXPERIMENT_NAME}_test_predictions_by_layer.csv",
)

manifest_json_path = STEP6_MANIFEST_JSON or os.path.join(
    STEP6_OUTPUT_DIR,
    f"{EXPERIMENT_NAME}_manifest.json",
)

os.makedirs(STEP6_OUTPUT_DIR, exist_ok=True)

with open(result_json_path, "w", encoding="utf-8") as f:
    json.dump(make_json_safe(full_result), f, ensure_ascii=False, indent=2)

layer_scores_df.to_csv(layer_scores_csv_path, index=False)
metrics_df.to_csv(metrics_csv_path, index=False)
recall_matrix_long_df.to_csv(recall_matrix_long_csv_path, index=False)
predictions_df.to_csv(predictions_csv_path, index=False)

manifest = {
    "experiment_id": EXPERIMENT_ID,
    "experiment_name": EXPERIMENT_NAME,
    "experiment_description": EXPERIMENT_DESCRIPTION,
    "script_family": SCRIPT_FAMILY,
    "model_name": MODEL_NAME,
    "model_tag": MODEL_TAG,
    "sample_families": SAMPLE_FAMILIES,
    "feature_key": FEATURE_KEY,
    "label_field": LABEL_FIELD,
    "keep_evidence_types": sorted(KEEP_EVIDENCE_TYPES),
    "output_files": {
        "full_result_json": result_json_path,
        "layer_scores_csv": layer_scores_csv_path,
        "per_layer_label_metrics_csv": metrics_csv_path,
        "recall_matrix_long_csv": recall_matrix_long_csv_path,
        "test_predictions_by_layer_csv": predictions_csv_path,
        "manifest_json": manifest_json_path,
    },
    "best_layer_summary": best_layer_summary,
    "num_layers_total": NUM_LAYERS,
    "num_layers_evaluated": len(LAYER_INDICES_EVALUATED),
    "layer_indices_evaluated": LAYER_INDICES_EVALUATED,
    "feature_dim": FEATURE_DIM,
    "feature_dim": FEATURE_DIM,
    "label_order": label_order,
    "scene_split_info": scene_split_info,
    "test_subsets": {
        name: int(len(indices))
        for name, indices in test_subsets.items()
    },
}

with open(manifest_json_path, "w", encoding="utf-8") as f:
    json.dump(make_json_safe(manifest), f, ensure_ascii=False, indent=2)

print("Saved Step 6 outputs:")
print("Full result JSON:", result_json_path)
print("Layer scores CSV:", layer_scores_csv_path)
print("Per-layer label metrics CSV:", metrics_csv_path)
print("Recall matrix long CSV:", recall_matrix_long_csv_path)
print("Test predictions by layer CSV:", predictions_csv_path)
print("Manifest JSON:", manifest_json_path)

print("\nBest layer summary:")
print(json.dumps(make_json_safe(best_layer_summary), indent=2, ensure_ascii=False))

if PRINT_TOP_LAYERS:
    sort_col = "overall_test_macro_f1"

    if sort_col not in layer_scores_df.columns:
        raise KeyError(
            f"Expected column {sort_col!r} in layer_scores_df. "
            f"Available columns: {list(layer_scores_df.columns)}"
        )

    print("\nTop layers by overall test macro-F1:")
    display(
        layer_scores_df.sort_values(
            by=sort_col,
            ascending=False,
        ).head(NUM_TOP_LAYERS_TO_PRINT)
    )

Saved Step 6 outputs:
Full result JSON: /content/drive/MyDrive/Colab Notebooks/linear_probe_full/data_prepare_step1_3/data/step6_probe/step6_pilot_full_ithor120_diverse_qwen2_5_3b_pair_explicit_direct_relation_ld_scene_split/step6_pilot_full_ithor120_diverse_qwen2_5_3b_pair_explicit_direct_relation_ld_scene_split_full_results.json
Layer scores CSV: /content/drive/MyDrive/Colab Notebooks/linear_probe_full/data_prepare_step1_3/data/step6_probe/step6_pilot_full_ithor120_diverse_qwen2_5_3b_pair_explicit_direct_relation_ld_scene_split/step6_pilot_full_ithor120_diverse_qwen2_5_3b_pair_explicit_direct_relation_ld_scene_split_layer_scores.csv
Per-layer label metrics CSV: /content/drive/MyDrive/Colab Notebooks/linear_probe_full/data_prepare_step1_3/data/step6_probe/step6_pilot_full_ithor120_diverse_qwen2_5_3b_pair_explicit_direct_relation_ld_scene_split/step6_pilot_full_ithor120_diverse_qwen2_5_3b_pair_explicit_direct_relation_ld_scene_split_per_layer_label_metrics.csv
Recall matrix long CSV: /

,experiment_name,experiment_id,script_family,layer,train_accuracy,train_macro_f1,explicit_direct_test_accuracy,explicit_direct_test_macro_f1,explicit_direct_num_examples,overall_test_accuracy,overall_test_macro_f1,overall_num_examples
5,step6_pilot_full_ithor120_diverse_qwen2_5_3b_p...,pair_explicit_direct_relation_ld,pair_relation_layerwise_logreg,5,0.978795,0.982360,0.851304,0.844544,4257,0.851304,0.844544,4257
3,step6_pilot_full_ithor120_diverse_qwen2_5_3b_p...,pair_explicit_direct_relation_ld,pair_relation_layerwise_logreg,3,0.974794,0.978770,0.848250,0.828642,4257,0.848250,0.828642,4257
2,step6_pilot_full_ithor120_diverse_qwen2_5_3b_p...,pair_explicit_direct_relation_ld,pair_relation_layerwise_logreg,2,0.971443,0.976121,0.844726,0.821016,4257,0.844726,0.821016,4257
4,step6_pilot_full_ithor120_diverse_qwen2_5_3b_p...,pair_explicit_direct_relation_ld,pair_relation_layerwise_logreg,4,0.978045,0.981519,0.837914,0.814498,4257,0.837914,0.814498,4257
1,step6_pilot_full_ithor120_diverse_qwen2_5_3b_p...,pair_explicit_direct_relation_ld,pair_relation_layerwise_logreg,1,0.962241,0.968365,0.832276,0.811616,4257,0.832276,0.811616,4257
6,step6_pilot_full_ithor120_diverse_qwen2_5_3b_p...,pair_explicit_direct_relation_ld,pair_relation_layerwise_logreg,6,0.981595,0.984859,0.834390,0.803717,4257,0.834390,0.803717,4257
7,step6_pilot_full_ithor120_diverse_qwen2_5_3b_p...,pair_explicit_direct_relation_ld,pair_relation_layerwise_logreg,7,0.983346,0.986540,0.821001,0.802813,4257,0.821001,0.802813,4257
22,step6_pilot_full_ithor120_diverse_qwen2_5_3b_p...,pair_explicit_direct_relation_ld,pair_relation_layerwise_logreg,22,0.993648,0.994780,0.814423,0.799439,4257,0.814423,0.799439,4257
20,step6_pilot_full_ithor120_diverse_qwen2_5_3b_p...,pair_explicit_direct_relation_ld,pair_relation_layerwise_logreg,20,0.992548,0.993908,0.815833,0.795779,4257,0.815833,0.795779,4257
10,step6_pilot_full_ithor120_diverse_qwen2_5_3b_p...,pair_explicit_direct_relation_ld,pair_relation_layerwise_logreg,10,0.983196,0.986417,0.800329,0.791363,4257,0.800329,0.791363,4257


In [ ]:
# Result integrity check

print("=" * 100)
print("Step 6 result integrity check")
print("=" * 100)

expected_output_files = {
    "full_result_json": result_json_path,
    "per_layer_label_metrics_csv": metrics_csv_path,
    "recall_matrix_long_csv": recall_matrix_long_csv_path,
    "layer_scores_csv": layer_scores_csv_path,
    "test_predictions_by_layer_csv": predictions_csv_path,
    "manifest_json": manifest_json_path,
}

missing_files = {
    name: path
    for name, path in expected_output_files.items()
    if not os.path.exists(path)
}

if missing_files:
    raise FileNotFoundError(
        "Some expected Step 6 output files are missing:\n"
        + json.dumps(missing_files, indent=2, ensure_ascii=False)
    )

print("All expected output files exist.")

layer_scores_df = pd.read_csv(layer_scores_csv_path)
metrics_df = pd.read_csv(metrics_csv_path)
recall_matrix_long_df = pd.read_csv(recall_matrix_long_csv_path)
predictions_df = pd.read_csv(predictions_csv_path)

required_layer_score_cols = {
    "experiment_name",
    "experiment_id",
    "script_family",
    "layer",
    "train_accuracy",
    "train_macro_f1",
    "overall_test_accuracy",
    "overall_test_macro_f1",
    "overall_num_examples",
}

# Add subset-specific columns dynamically.
for subset_name in sorted(test_subsets.keys()):
    required_layer_score_cols.update({
        f"{subset_name}_test_accuracy",
        f"{subset_name}_test_macro_f1",
        f"{subset_name}_num_examples",
    })

required_metrics_cols = {
    "experiment_name",
    "experiment_id",
    "script_family",
    "subset",
    "layer",
    "label",
    "precision",
    "recall",
    "f1_score",
    "support",
    "accuracy",
    "macro_f1",
    "num_examples",
}

required_recall_matrix_cols = {
    "experiment_name",
    "experiment_id",
    "script_family",
    "subset",
    "layer",
    "label",
    "metric",
    "value",
}

required_prediction_cols = {
    "experiment_name",
    "experiment_id",
    "script_family",
    "layer",
    "sample_id",
    "pair_group_id",
    "scene",
    "paragraph_id",
    "evidence_type",
    "probe_direction_relative_to_text",
    "is_inverse_of_text_relation",
    "true_label",
    "pred_label",
    "is_correct",
    "text_relation_label",
    "text_subject_uid",
    "text_object_uid",
    "subject_uid",
    "object_uid",
    "subject_alias",
    "object_alias",
    "evidence_sentence",
    "text",
    "_step6_sample_family",
    "_source_step5_pt_file",
    "_source_step5_input_dir",
}

def check_required_columns(df, required_cols, file_label):
    missing = sorted(required_cols - set(df.columns))

    if missing:
        raise ValueError(
            f"{file_label} is missing required columns: {missing}"
        )

    return {
        "file_label": file_label,
        "num_rows": int(len(df)),
        "num_columns": int(len(df.columns)),
    }


column_check_summary = {
    "layer_scores": check_required_columns(
        layer_scores_df,
        required_layer_score_cols,
        "layer_scores_csv",
    ),
    "per_layer_label_metrics": check_required_columns(
        metrics_df,
        required_metrics_cols,
        "per_layer_label_metrics_csv",
    ),
    "recall_matrix_long": check_required_columns(
        recall_matrix_long_df,
        required_recall_matrix_cols,
        "recall_matrix_long_csv",
    ),
    "test_predictions_by_layer": check_required_columns(
        predictions_df,
        required_prediction_cols,
        "test_predictions_by_layer_csv",
    ),
}

print("\nColumn checks passed.")
print(json.dumps(column_check_summary, indent=2, ensure_ascii=False))


expected_subsets = set(test_subsets.keys())

metrics_subsets = set(metrics_df["subset"].dropna().unique())
recall_subsets = set(recall_matrix_long_df["subset"].dropna().unique())

missing_metrics_subsets = expected_subsets - metrics_subsets
missing_recall_subsets = expected_subsets - recall_subsets

if missing_metrics_subsets:
    raise ValueError(
        f"per_layer_label_metrics is missing subsets: {sorted(missing_metrics_subsets)}"
    )

if missing_recall_subsets:
    raise ValueError(
        f"recall_matrix_long is missing subsets: {sorted(missing_recall_subsets)}"
    )

unexpected_metrics_subsets = metrics_subsets - expected_subsets
unexpected_recall_subsets = recall_subsets - expected_subsets

if unexpected_metrics_subsets:
    raise ValueError(
        f"per_layer_label_metrics has unexpected subsets: {sorted(unexpected_metrics_subsets)}"
    )

if unexpected_recall_subsets:
    raise ValueError(
        f"recall_matrix_long has unexpected subsets: {sorted(unexpected_recall_subsets)}"
    )

print("\nSubset checks passed.")
print("Subsets in per_layer_label_metrics:", sorted(metrics_subsets))
print("Subsets in recall_matrix_long:", sorted(recall_subsets))

layer_scores_layers = sorted(layer_scores_df["layer"].unique().tolist())
metrics_layers = sorted(metrics_df["layer"].unique().tolist())
recall_layers = sorted(recall_matrix_long_df["layer"].unique().tolist())
prediction_layers = sorted(predictions_df["layer"].unique().tolist())

expected_layers = list(LAYER_INDICES_EVALUATED)

if layer_scores_layers != expected_layers:
    raise ValueError(
        "layer_scores does not cover all layers. "
        f"Expected={expected_layers}, got={layer_scores_layers}"
    )

if metrics_layers != expected_layers:
    raise ValueError(
        "per_layer_label_metrics does not cover all layers. "
        f"Expected={expected_layers}, got={metrics_layers}"
    )

if recall_layers != expected_layers:
    raise ValueError(
        "recall_matrix_long does not cover all layers. "
        f"Expected={expected_layers}, got={recall_layers}"
    )

if prediction_layers != expected_layers:
    raise ValueError(
        "test_predictions_by_layer does not cover all layers. "
        f"Expected={expected_layers}, got={prediction_layers}"
    )

print("\nLayer coverage checks passed.")
print("Number of layers:", NUM_LAYERS)


prediction_evidence_counts = dict(Counter(predictions_df["evidence_type"]))

print("\nPrediction evidence type counts:")
print(json.dumps(prediction_evidence_counts, indent=2, ensure_ascii=False))

unexpected_prediction_evidence = set(prediction_evidence_counts.keys()) - KEEP_EVIDENCE_TYPES

if unexpected_prediction_evidence:
    raise ValueError(
        "Predictions contain evidence types not allowed by STEP6_KEEP_EVIDENCE_TYPES:\n"
        + json.dumps(prediction_evidence_counts, indent=2, ensure_ascii=False)
    )


with open(result_json_path, "r", encoding="utf-8") as f:
    saved_result = json.load(f)

required_json_keys = {
    "experiment_id",
    "experiment_name",
    "description",
    "step6_config",
    "source_step5_files",
    "source_file_summaries",
    "merged_record_summary",
    "scene_split_info",
    "label_order",
    "num_layers_total",
    "num_layers_evaluated",
    "layer_indices_evaluated",
    "feature_dim",
    "test_subsets",
    "best_layer_summary",
    "results_by_layer",
    "per_layer_label_metrics",
    "recall_matrix_long",
}

missing_json_keys = sorted(required_json_keys - set(saved_result.keys()))

if missing_json_keys:
    raise ValueError(
        f"Full result JSON is missing required keys: {missing_json_keys}"
    )

print("\nFull result JSON key check passed.")


integrity_summary = {

    "use_local_content_cache": USE_LOCAL_CONTENT_CACHE,
    "local_content_cache_root": LOCAL_CONTENT_CACHE_ROOT if USE_LOCAL_CONTENT_CACHE else None,

    "num_layers_total": NUM_LAYERS,
    "num_layers_evaluated": len(LAYER_INDICES_EVALUATED),
    "layer_indices_evaluated": LAYER_INDICES_EVALUATED,

    "experiment_id": EXPERIMENT_ID,
    "experiment_name": EXPERIMENT_NAME,
    "experiment_short_name": EXPERIMENT_SHORT_NAME,
    "script_family": SCRIPT_FAMILY,

    "model_name": MODEL_NAME,
    "model_tag": MODEL_TAG,
    "feature_key": FEATURE_KEY,
    "label_field": LABEL_FIELD,
    "sample_families": SAMPLE_FAMILIES,
    "keep_evidence_types": sorted(KEEP_EVIDENCE_TYPES),

    "num_layers": NUM_LAYERS,
    "feature_dim": FEATURE_DIM,

    "expected_output_files": {
        key: os.path.basename(value)
        for key, value in expected_output_files.items()
    },

    "column_check_summary": column_check_summary,

    "subsets": {
        "expected": sorted(expected_subsets),
        "per_layer_label_metrics": sorted(metrics_subsets),
        "recall_matrix_long": sorted(recall_subsets),
    },

    "layer_coverage": {
        "num_layers": NUM_LAYERS,
        "layer_scores_layers": layer_scores_layers,
        "metrics_layers": metrics_layers,
        "recall_layers": recall_layers,
        "prediction_layers": prediction_layers,
    },

    "prediction_evidence_type_counts": prediction_evidence_counts,

    "json_key_check": {
        "required_keys": sorted(required_json_keys),
        "missing_keys": missing_json_keys,
    },

    "scene_split_info": scene_split_info,
    "best_layer_summary": best_layer_summary,
    "label_order": label_order,
}

integrity_summary_path = os.path.join(
    STEP6_OUTPUT_DIR,
    f"{EXPERIMENT_NAME}_integrity_summary.json",
)

with open(integrity_summary_path, "w", encoding="utf-8") as f:
    json.dump(make_json_safe(integrity_summary), f, ensure_ascii=False, indent=2)

print("\nIntegrity summary saved:")
print(integrity_summary_path)

print("\nStep 6 outputs are complete.")

Step 6 result integrity check
All expected output files exist.

Column checks passed.
{
  "layer_scores": {
    "file_label": "layer_scores_csv",
    "num_rows": 37,
    "num_columns": 12
  },
  "per_layer_label_metrics": {
    "file_label": "per_layer_label_metrics_csv",
    "num_rows": 666,
    "num_columns": 13
  },
  "recall_matrix_long": {
    "file_label": "recall_matrix_long_csv",
    "num_rows": 666,
    "num_columns": 8
  },
  "test_predictions_by_layer": {
    "file_label": "test_predictions_by_layer_csv",
    "num_rows": 157509,
    "num_columns": 26
  }
}

Subset checks passed.
Subsets in per_layer_label_metrics: ['explicit_direct', 'overall']
Subsets in recall_matrix_long: ['explicit_direct', 'overall']

Layer coverage checks passed.
Number of layers: 37

Prediction evidence type counts:
{
  "explicit_direct": 157509
}

Full result JSON key check passed.

Integrity summary saved:
/content/drive/MyDrive/Colab Notebooks/linear_probe_full/data_prepare_step1_3/data/step6_probe

In [ ]:
from google.colab import files

output_files = sorted(os.listdir(STEP6_OUTPUT_DIR))

if not output_files:
    raise FileNotFoundError(
        f"No files found in STEP6_OUTPUT_DIR: {STEP6_OUTPUT_DIR}"
    )

zip_base = f"/content/{EXPERIMENT_NAME}"

zip_path = shutil.make_archive(
    base_name=zip_base,
    format="zip",
    root_dir=STEP6_OUTPUT_DIR,
)

print("Created zip:", zip_path)
print("STEP6_OUTPUT_DIR:", STEP6_OUTPUT_DIR)
print("Number of files included:", len(output_files))

print("\nFiles included:")
for name in output_files:
    print("-", name)

files.download(zip_path)

Created zip: /content/step6_pilot_full_ithor120_diverse_qwen2_5_3b_pair_explicit_direct_relation_ld_scene_split.zip
STEP6_OUTPUT_DIR: /content/drive/MyDrive/Colab Notebooks/linear_probe_full/data_prepare_step1_3/data/step6_probe/step6_pilot_full_ithor120_diverse_qwen2_5_3b_pair_explicit_direct_relation_ld_scene_split
Number of files included: 7

Files included:
- step6_pilot_full_ithor120_diverse_qwen2_5_3b_pair_explicit_direct_relation_ld_scene_split_full_results.json
- step6_pilot_full_ithor120_diverse_qwen2_5_3b_pair_explicit_direct_relation_ld_scene_split_integrity_summary.json
- step6_pilot_full_ithor120_diverse_qwen2_5_3b_pair_explicit_direct_relation_ld_scene_split_layer_scores.csv
- step6_pilot_full_ithor120_diverse_qwen2_5_3b_pair_explicit_direct_relation_ld_scene_split_manifest.json
- step6_pilot_full_ithor120_diverse_qwen2_5_3b_pair_explicit_direct_relation_ld_scene_split_per_layer_label_metrics.csv
- step6_pilot_full_ithor120_diverse_qwen2_5_3b_pair_explicit_direct_relation

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>